# Netflix Catalog — SQL Solutions & Findings

A reproducible notebook for the 15 Netflix catalog questions. It runs PostgreSQL queries against the `netflix` table and derives findings from actual query outputs.

**Prerequisites:** PostgreSQL and a populated `netflix` table. Set `PGHOST`, `PGPORT`, `PGUSER`, `PGPASSWORD`, `PGDATABASE`. The repository's `netflix_titles.csv` was reported as an empty placeholder in the current project notes, so restore the real dataset before interpreting results.

## 1. Connect and inspect source data

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
required = ["PGHOST", "PGUSER", "PGPASSWORD", "PGDATABASE"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError("Set PostgreSQL variables: " + ", ".join(missing))
engine = create_engine(
    f"postgresql+psycopg2://{os.environ['PGUSER']}:{os.environ['PGPASSWORD']}"
    f"@{os.environ['PGHOST']}:{os.getenv('PGPORT','5432')}/{os.environ['PGDATABASE']}"
)
def run_query(sql):
    return pd.read_sql_query(text(sql), engine)
source = run_query("""
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT show_id) AS distinct_show_ids,
       MIN(release_year) AS first_release_year,
       MAX(release_year) AS last_release_year
FROM netflix
""")
display(source)
if source.empty or int(source.loc[0,"row_count"]) == 0:
    raise ValueError("netflix is empty. Load the dataset before running the case study.")

## 2. Q1 — Movies vs TV Shows

In [ ]:
q1=run_query("""
SELECT type, COUNT(*) AS content_count
FROM netflix
GROUP BY type
ORDER BY content_count DESC
""")
display(q1)
if not q1.empty:
    total=q1["content_count"].sum()
    display(q1.assign(share_pct=(100*q1["content_count"]/total).round(2)))
    print("Finding: this shows catalog composition by title type; it does not measure viewing hours or popularity.")

## 3. Q2 — Most common rating by content type

In [ ]:
q2=run_query("""
WITH counts AS (
 SELECT type, rating, COUNT(*) AS rating_count
 FROM netflix
 WHERE rating IS NOT NULL
 GROUP BY type, rating
), ranked AS (
 SELECT *, RANK() OVER(PARTITION BY type ORDER BY rating_count DESC) AS rnk
 FROM counts
)
SELECT type, rating AS most_frequent_rating, rating_count
FROM ranked WHERE rnk=1
ORDER BY type
""")
display(q2)
print("Finding: tied ratings are retained; missing ratings are excluded from the ranking.")

## 4. Q3 — Titles released in 2020

In [ ]:
q3=run_query("SELECT show_id, title, type, release_year, country FROM netflix WHERE release_year=2020 ORDER BY title")
display(q3)
print(f"Finding: {len(q3):,} catalog rows have release_year=2020.")

## 5. Q4 — Top five countries by title count

The country field is multi-valued. Split and trim country names so a title listing multiple countries contributes once to each listed country.

In [ ]:
q4=run_query("""
WITH country_rows AS (
 SELECT n.show_id, BTRIM(c.country_name) AS country
 FROM netflix n
 CROSS JOIN LATERAL UNNEST(STRING_TO_ARRAY(n.country, ',')) AS c(country_name)
 WHERE n.country IS NOT NULL AND BTRIM(c.country_name) <> ''
)
SELECT country, COUNT(DISTINCT show_id) AS title_count
FROM country_rows
GROUP BY country
ORDER BY title_count DESC, country
LIMIT 5
""")
display(q4)
if not q4.empty:
    print(f"Finding: {q4.iloc[0]['country']} has the most listed titles in this dataset ({int(q4.iloc[0]['title_count']):,}). Multi-country titles can count for each country listed.")

## 6. Q5 — Longest movies

In [ ]:
q5=run_query("""
SELECT show_id, title, duration,
       NULLIF(SPLIT_PART(duration, ' ', 1), '')::INTEGER AS duration_minutes
FROM netflix
WHERE type='Movie' AND duration ~ '^[0-9]+ min$'
ORDER BY duration_minutes DESC, title
LIMIT 10
""")
display(q5)
if not q5.empty:
    r=q5.iloc[0]
    print(f"Finding: longest listed movie is {r['title']} at {int(r['duration_minutes'])} minutes (among rows with parseable minute durations).")

## 7. Q6 — Content added in the last five years

This is based on date_added, not release_year. Invalid or missing date strings are excluded safely.

In [ ]:
q6=run_query("""
SELECT show_id, title, type, date_added
FROM netflix
WHERE TO_DATE(NULLIF(BTRIM(date_added), ''), 'Month DD, YYYY')
      >= CURRENT_DATE - INTERVAL '5 years'
ORDER BY TO_DATE(NULLIF(BTRIM(date_added), ''), 'Month DD, YYYY') DESC NULLS LAST
""")
display(q6)
print(f"Finding: {len(q6):,} rows meet the five-year date_added filter as of the database's current date.")

## 8. Q7 — Titles credited to Rajiv Chilaka

In [ ]:
q7=run_query("""
SELECT n.show_id, n.title, n.type, n.release_year
FROM netflix n
WHERE EXISTS (
 SELECT 1 FROM UNNEST(STRING_TO_ARRAY(COALESCE(n.director,''), ',')) d(name)
 WHERE BTRIM(d.name) = 'Rajiv Chilaka'
)
ORDER BY n.release_year DESC, n.title
""")
display(q7)
print(f"Finding: {len(q7):,} titles list Rajiv Chilaka in the director field.")

## 9. Q8 — TV shows with more than five seasons

In [ ]:
q8=run_query("""
SELECT show_id, title, duration,
       SPLIT_PART(duration, ' ', 1)::INTEGER AS seasons
FROM netflix
WHERE type='TV Show' AND duration ~ '^[0-9]+ Seasons?$'
  AND SPLIT_PART(duration, ' ', 1)::INTEGER > 5
ORDER BY seasons DESC, title
""")
display(q8)
print(f"Finding: {len(q8):,} TV shows list more than five seasons.")

## 10. Q9 — Content count by genre

A title with multiple listed genres is counted once under each genre.

In [ ]:
q9=run_query("""
WITH genre_rows AS (
 SELECT n.show_id, BTRIM(g.genre_name) AS genre
 FROM netflix n
 CROSS JOIN LATERAL UNNEST(STRING_TO_ARRAY(n.listed_in, ',')) g(genre_name)
 WHERE n.listed_in IS NOT NULL AND BTRIM(g.genre_name) <> ''
)
SELECT genre, COUNT(DISTINCT show_id) AS total_content
FROM genre_rows
GROUP BY genre
ORDER BY total_content DESC, genre
""")
display(q9)
if not q9.empty:
    print(f"Finding: {q9.iloc[0]['genre']} is the most represented listed genre ({int(q9.iloc[0]['total_content']):,} titles).")

## 11. Q10 — Top five release years by India's catalog count

The original query label said “average” but the calculation was a share of India titles. This solution instead reports annual title counts and their share of India-linked titles. Country matching splits the multi-valued country field.

In [ ]:
q10=run_query("""
WITH india_titles AS (
 SELECT DISTINCT n.show_id, n.release_year
 FROM netflix n
 CROSS JOIN LATERAL UNNEST(STRING_TO_ARRAY(COALESCE(n.country,''), ',')) c(country_name)
 WHERE BTRIM(c.country_name)='India' AND n.release_year IS NOT NULL
), annual AS (
 SELECT release_year, COUNT(*) AS title_count FROM india_titles GROUP BY release_year
)
SELECT release_year, title_count,
       ROUND(100.0*title_count/NULLIF(SUM(title_count) OVER (),0),2) AS share_of_india_titles_pct
FROM annual
ORDER BY title_count DESC, release_year DESC
LIMIT 5
""")
display(q10)
print("Finding: these are the five release years with the most India-linked titles in the catalog; catalog counts are not a measure of audience demand.")

## 12. Q11 — Documentaries

In [ ]:
q11=run_query("""
SELECT show_id, title, type, listed_in
FROM netflix
WHERE listed_in ILIKE '%Documentaries%'
ORDER BY title
""")
display(q11)
print(f"Finding: {len(q11):,} rows have a listed_in value containing the Documentary label.")

## 13. Q12 — Content without a director

In [ ]:
q12=run_query("""
SELECT type, COUNT(*) AS titles_without_director
FROM netflix
WHERE director IS NULL OR BTRIM(director)=''
GROUP BY type
ORDER BY titles_without_director DESC
""")
display(q12)
print("Finding: titles without director metadata indicate a catalog-completeness gap, not necessarily a missing real-world director credit.")

## 14. Q13 — Salman Khan movies released in the last ten years

In [ ]:
q13=run_query("""
SELECT show_id, title, release_year, casts
FROM netflix
WHERE type='Movie'
  AND casts ILIKE '%Salman Khan%'
  AND release_year > EXTRACT(YEAR FROM CURRENT_DATE)::INTEGER - 10
ORDER BY release_year DESC, title
""")
display(q13)
print(f"Finding: {len(q13):,} movie rows match the cast-text and release-year filter. This is a text match, not an independently verified filmography.")

## 15. Q14 — Top ten actors in India-produced titles

In [ ]:
q14=run_query("""
WITH india_titles AS (
 SELECT DISTINCT n.show_id, n.casts
 FROM netflix n
 CROSS JOIN LATERAL UNNEST(STRING_TO_ARRAY(COALESCE(n.country,''), ',')) c(country_name)
 WHERE BTRIM(c.country_name)='India'
), actors AS (
 SELECT i.show_id, BTRIM(a.actor_name) AS actor
 FROM india_titles i
 CROSS JOIN LATERAL UNNEST(STRING_TO_ARRAY(COALESCE(i.casts,''), ',')) a(actor_name)
 WHERE BTRIM(a.actor_name)<>''
)
SELECT actor, COUNT(DISTINCT show_id) AS title_count
FROM actors
GROUP BY actor
ORDER BY title_count DESC, actor
LIMIT 10
""")
display(q14)
print("Finding: this ranks cast names by appearances in India-linked titles. Credits in the source are not a complete or authoritative filmography.")

## 16. Q15 — Description keyword classification

This is a simple keyword rule for practice, not a content-safety or suitability classifier.

In [ ]:
q15=run_query("""
SELECT category, type, COUNT(*) AS content_count
FROM (
 SELECT type,
        CASE WHEN description ILIKE '%kill%' OR description ILIKE '%violence%'
             THEN 'Keyword match' ELSE 'No keyword match' END AS category
 FROM netflix
) x
GROUP BY category, type
ORDER BY type, category
""")
display(q15)
print("Finding: keyword matches are a crude text flag; they can miss context and must not be interpreted as a reliable quality or safety label.")

## Executive summary
Use the query result tables above to report catalog composition, country/genre representation, long-form content, missing metadata and description keyword prevalence. Before publishing any numeric finding, confirm that the source CSV was restored and the notebook ran successfully against the current database.